# Upload Trained Model to Hugging Face Hub
**Run this in a SEPARATE Kaggle notebook after training completes.**

- No GPU needed — set Accelerator to None
- Internet must be ON
- Add your training notebook as Input data (right panel → Input → Your Work)

## Cell 1 — Find your training output files

In [ ]:
import os

# This prints all files available from your training notebook output
# Look for the path containing marine_trash_v2/
print('Available input files:')
for root, dirs, files in os.walk('/kaggle/input/'):
    for file in files:
        print(os.path.join(root, file))

## Cell 2 — Set your paths

After running Cell 1, look at the output and find the path that contains `best.pt`.
Copy that folder path and paste it as `INPUT_DIR` below.

In [ ]:
# ── EDIT THESE TWO LINES ONLY ────────────────────────────────
HF_USERNAME = 'YOUR_HF_USERNAME'          # your HF username
INPUT_DIR   = '/kaggle/input/YOUR_TRAINING_NOTEBOOK_NAME/marine_trash_v2'
# ─────────────────────────────────────────────────────────────

REPO_ID   = f'{HF_USERNAME}/yolov8m-marine-trash'
MODEL_PT  = f'{INPUT_DIR}/best.pt'

# Verify files exist
files_to_upload = [
    f'{INPUT_DIR}/best.pt',
    f'{INPUT_DIR}/training_curves.png',
    f'{INPUT_DIR}/confusion_matrix.png',
    f'{INPUT_DIR}/results.png',
    f'{INPUT_DIR}/val_batch0_pred.jpg',
    f'{INPUT_DIR}/class_distribution.png',
]

print('Checking files...')
for f in files_to_upload:
    status = 'FOUND' if os.path.exists(f) else 'MISSING'
    print(f'  {status} : {os.path.basename(f)}')

print(f'\nRepo ID  : {REPO_ID}')
print(f'Model    : {MODEL_PT}')

## Cell 3 — Install Hugging Face Hub

In [ ]:
!pip install huggingface_hub -q

## Cell 4 — Login to Hugging Face

Uses Kaggle Secrets so your token is never visible in the notebook.

Before running: go to **Add-ons → Secrets → Add new secret**
- Name: `HF_TOKEN`
- Value: your token from huggingface.co/settings/tokens (starts with hf_...)

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret('HF_TOKEN')
login(token=token)
print('Logged in to Hugging Face.')

## Cell 5 — Create repo and upload all files

In [ ]:
from huggingface_hub import HfApi, create_repo

# Create model repo (skips if already exists)
create_repo(REPO_ID, repo_type='model', exist_ok=True)
print(f'Repo ready: https://huggingface.co/{REPO_ID}')

api = HfApi()

# Upload each file
upload_map = {
    f'{INPUT_DIR}/best.pt'               : 'yolov8m_marine_best.pt',
    f'{INPUT_DIR}/training_curves.png'   : 'training_curves.png',
    f'{INPUT_DIR}/confusion_matrix.png'  : 'confusion_matrix.png',
    f'{INPUT_DIR}/results.png'           : 'results.png',
    f'{INPUT_DIR}/val_batch0_pred.jpg'   : 'val_batch0_pred.jpg',
    f'{INPUT_DIR}/class_distribution.png': 'class_distribution.png',
}

for local_path, remote_name in upload_map.items():
    if os.path.exists(local_path):
        api.upload_file(
            path_or_fileobj=local_path,
            path_in_repo=remote_name,
            repo_id=REPO_ID,
            repo_type='model',
        )
        print(f'Uploaded: {remote_name}')
    else:
        print(f'Skipped (not found): {remote_name}')

## Cell 6 — Write model card (README on HF)

In [ ]:
# Read final mAP from results.csv if available
import pandas as pd

results_csv = f'{INPUT_DIR}/results.csv' if os.path.exists(f'{INPUT_DIR}/results.csv') else None
det_map50   = 'N/A'
seg_map50   = 'N/A'

if results_csv and os.path.exists(results_csv):
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    if 'metrics/mAP50(B)' in df.columns:
        det_map50 = f"{df['metrics/mAP50(B)'].max():.1%}"
    if 'metrics/mAP50(M)' in df.columns:
        seg_map50 = f"{df['metrics/mAP50(M)'].max():.1%}"

model_card = f"""---
license: mit
tags:
- yolov8
- object-detection
- instance-segmentation
- underwater
- marine-trash
- computer-vision
datasets:
- trashcan-1.0
metrics:
- mAP
---

# YOLOv8m-seg — Marine Trash Detection

Fine-tuned YOLOv8m-seg on the [TrashCan 1.0](https://conservancy.umn.edu/handle/11299/214865)
dataset for underwater marine debris detection and segmentation.

**Project:** Life Under Water | VIPS-TC, GGSIPU
**Author:** Krishna Jaiswal

## Results

| Metric | Score |
|--------|-------|
| mAP50 (detection) | {det_map50} |
| mAP50 (segmentation) | {seg_map50} |
| Classes | 16 |
| Training images | 6,008 |
| Epochs | 75 |

## Training curves
![Training curves](training_curves.png)

## Sample predictions
![Predictions](val_batch0_pred.jpg)

## Classes
Trash: `trash_plastic`, `trash_metal`, `trash_fabric`, `trash_fishing_gear`,
`trash_rubber`, `trash_wood`, `trash_paper`, `trash_etc`

Marine life: `animal_fish`, `animal_starfish`, `animal_shells`, `animal_crab`,
`animal_eel`, `animal_etc`, `plant`, `rov`

## Usage

```python
from ultralytics import YOLO
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(
    repo_id='{REPO_ID}',
    filename='yolov8m_marine_best.pt'
)
model = YOLO(model_path)
results = model.predict('underwater_image.jpg', task='segment')
results[0].show()
```

## Live Demo
[Hugging Face Spaces](https://huggingface.co/spaces/{HF_USERNAME}/marine-trash-detection)

## Training config
- Optimizer: SGD with cosine LR decay (lr0=0.01 → 0.001)
- Augmentation: mosaic, mixup=0.1, copy_paste=0.1, HSV jitter
- Image size: 640×640 | Batch: 16 | Platform: Kaggle P100 GPU
"""

with open('/kaggle/working/README.md', 'w') as f:
    f.write(model_card)

api.upload_file(
    path_or_fileobj='/kaggle/working/README.md',
    path_in_repo='README.md',
    repo_id=REPO_ID,
    repo_type='model',
)

print('Model card uploaded.')
print(f'\nYour model is live at:')
print(f'https://huggingface.co/{REPO_ID}')
print(f'\nYour demo Space:')
print(f'https://huggingface.co/spaces/{HF_USERNAME}/marine-trash-detection')